# Лабораторная работа №6: Поиск экстремума функции двух переменных с помощью ГА с бинарным представлением особей

**Цель работы:** реализовать генетический алгоритм с бинарным кодированием для нахождения минимума функции двух переменных.

**Задачи:**
1. Реализовать бинарное кодирование/декодирование двух вещественных переменных $x_1, x_2 \in [-100, 100]$.
2. Реализовать операторы ГА: турнирную селекцию, равномерный кроссовер, инверсионную мутацию.
3. Провести поиск минимума функции Шаффера F7.
4. Сравнить стратегии редукции: элитарная, полная замена, случайная замена.
5. Исследовать влияние гиперпараметров на качество оптимизации.

**Вариант 3.** Функция Шаффера F7:

$$f(x_1, x_2) = (x_1^2 + x_2^2)^{0.25} \cdot \left[\sin^2\!\left(50 \cdot (x_1^2 + x_2^2)^{0.1}\right) + 1\right], \quad x_i \in [-100,\; 100]$$

Глобальный минимум: $f(0, 0) = 0$.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

OUTPUTS = Path("report/outputs")
OUTPUTS.mkdir(exist_ok=True)

np.random.seed(42)

## 1. Целевая функция — Schaffer F7

$$f(x_1, x_2) = (x_1^2 + x_2^2)^{0.25} \cdot \left[\sin^2\!\left(50 \cdot (x_1^2 + x_2^2)^{0.1}\right) + 1\right]$$

Область определения: $x_1, x_2 \in [-100, 100]$. Глобальный минимум: $f(0, 0) = 0$.

Функция имеет множество локальных минимумов, расположенных концентрическими кольцами вокруг начала координат, что делает её сложным тестом для оптимизационных алгоритмов.

In [ ]:
A, B = -100.0, 100.0


def schaffer_f7(x1: np.ndarray, x2: np.ndarray) -> np.ndarray:
    r2 = x1**2 + x2**2
    return np.where(
        r2 == 0, 0.0,
        r2**0.25 * (np.sin(50.0 * r2**0.1) ** 2 + 1.0),
    )


def schaffer_f7_vec(pop: np.ndarray) -> np.ndarray:
    """Vectorised version: pop shape (N, 2) -> (N,)."""
    return schaffer_f7(pop[:, 0], pop[:, 1])

In [ ]:
grid_res = 300
x_grid = np.linspace(A, B, grid_res)
X1, X2 = np.meshgrid(x_grid, x_grid)
Z = schaffer_f7(X1, X2)

fig = go.Figure(data=go.Contour(
    x=x_grid, y=x_grid, z=Z,
    contours=dict(start=0, end=20, size=1),
    colorscale="Viridis",
    colorbar=dict(title="f(x₁, x₂)"),
))
fig.add_trace(go.Scatter(
    x=[0], y=[0], mode="markers",
    marker=dict(size=12, color="red", symbol="star"),
    name="Глобальный минимум (0, 0)",
))
fig.update_layout(
    title="Функция Шаффера F7: линии уровня",
    xaxis_title="x₁", yaxis_title="x₂",
    width=700, height=600,
    template="plotly_white",
)
fig.write_html(OUTPUTS / "01_schaffer_f7_surface.html")
fig.show()

## 2. Бинарное кодирование двух переменных

Каждая переменная $x_i \in [a, b]$ кодируется бинарной строкой длины $L$. Хромосома — конкатенация двух строк (общая длина $2L$).

Декодирование:

$$x_i = a + \text{decimal}(\text{bits}_i) \cdot \frac{b - a}{2^L - 1}$$

При $L = 20$ бит точность дискретизации:

$$\Delta x = \frac{200}{2^{20} - 1} \approx 1.9 \times 10^{-4}$$

In [ ]:
L_BITS = 20
CHROM_LEN = 2 * L_BITS
MAX_VAL = 2**L_BITS - 1


def decode_chromosome(chroms: np.ndarray) -> np.ndarray:
    """Decode binary chromosomes (N, 2*L) to real values (N, 2)."""
    powers = 2 ** np.arange(L_BITS - 1, -1, -1)
    bits1 = chroms[:, :L_BITS]
    bits2 = chroms[:, L_BITS:]
    dec1 = bits1 @ powers
    dec2 = bits2 @ powers
    x1 = A + dec1 * (B - A) / MAX_VAL
    x2 = A + dec2 * (B - A) / MAX_VAL
    return np.column_stack([x1, x2])


def encode_real(x: np.ndarray) -> np.ndarray:
    """Encode real values (N, 2) to binary chromosomes (N, 2*L)."""
    dec = np.round((x - A) / (B - A) * MAX_VAL).astype(int)
    dec = np.clip(dec, 0, MAX_VAL)
    bits = np.zeros((x.shape[0], CHROM_LEN), dtype=np.int8)
    for i in range(L_BITS):
        bits[:, L_BITS - 1 - i] = (dec[:, 0] >> i) & 1
        bits[:, CHROM_LEN - 1 - i] = (dec[:, 1] >> i) & 1
    return bits


test_vals = np.array([[-100.0, 100.0], [0.0, 0.0], [50.0, -50.0]])
encoded = encode_real(test_vals)
decoded = decode_chromosome(encoded)
print("Проверка кодирования/декодирования:")
for orig, dec in zip(test_vals, decoded):
    print(f"  Исходное: ({orig[0]:8.3f}, {orig[1]:8.3f})  →  Декод.: ({dec[0]:8.3f}, {dec[1]:8.3f})")

## 3. Операторы генетического алгоритма

### Турнирная селекция
Из популяции случайно выбираются $k$ особей; лучшая из них становится родителем.

### Равномерный кроссовер
Для каждого бита хромосомы независимо выбирается, от какого из двух родителей он будет унаследован.

### Инверсионная мутация
Случайный сегмент хромосомы разворачивается (reverse). Это сохраняет набор аллелей, но меняет их расположение.

In [ ]:
class BinaryGA:
    """Genetic Algorithm with binary encoding for minimisation of f(x1, x2)."""

    def __init__(
        self,
        fitness_fn,
        a: float = -100.0,
        b: float = 100.0,
        bits_per_var: int = 20,
        pop_size: int = 80,
        n_generations: int = 200,
        crossover_prob: float = 0.85,
        mutation_prob: float = 0.05,
        tournament_k: int = 3,
        reduction: str = "elitist",
        seed: int = 42,
    ):
        self.fitness_fn = fitness_fn
        self.a, self.b = a, b
        self.L = bits_per_var
        self.chrom_len = 2 * bits_per_var
        self.max_val = 2**bits_per_var - 1
        self.pop_size = pop_size
        self.n_generations = n_generations
        self.crossover_prob = crossover_prob
        self.mutation_prob = mutation_prob
        self.tournament_k = tournament_k
        self.reduction = reduction
        self.rng = np.random.RandomState(seed)

        self.population: np.ndarray | None = None
        self.fitness: np.ndarray | None = None
        self.best_history: list[float] = []
        self.mean_history: list[float] = []
        self.best_individual_history: list[np.ndarray] = []
        self.population_history: list[np.ndarray] = []

    def _decode(self, chroms: np.ndarray) -> np.ndarray:
        powers = 2 ** np.arange(self.L - 1, -1, -1)
        dec1 = chroms[:, :self.L] @ powers
        dec2 = chroms[:, self.L:] @ powers
        x1 = self.a + dec1 * (self.b - self.a) / self.max_val
        x2 = self.a + dec2 * (self.b - self.a) / self.max_val
        return np.column_stack([x1, x2])

    def _evaluate(self, chroms: np.ndarray) -> np.ndarray:
        reals = self._decode(chroms)
        return self.fitness_fn(reals)

    def _init_population(self) -> None:
        self.population = self.rng.randint(0, 2, size=(self.pop_size, self.chrom_len)).astype(np.int8)
        self.fitness = self._evaluate(self.population)

    def _tournament_selection(self) -> int:
        indices = self.rng.randint(0, len(self.population), size=self.tournament_k)
        return indices[np.argmin(self.fitness[indices])]

    def _uniform_crossover(self, p1: np.ndarray, p2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        mask = self.rng.randint(0, 2, size=self.chrom_len).astype(np.int8)
        child1 = np.where(mask, p1, p2)
        child2 = np.where(mask, p2, p1)
        return child1.astype(np.int8), child2.astype(np.int8)

    def _inversion_mutation(self, chrom: np.ndarray) -> np.ndarray:
        pts = sorted(self.rng.randint(0, self.chrom_len, size=2))
        if pts[0] == pts[1]:
            pts[1] = min(pts[1] + 1, self.chrom_len - 1)
        chrom = chrom.copy()
        chrom[pts[0]:pts[1] + 1] = chrom[pts[0]:pts[1] + 1][::-1]
        return chrom

    def _reduce(self, parents: np.ndarray, parent_fit: np.ndarray,
                offspring: np.ndarray, offspring_fit: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        if self.reduction == "elitist":
            combined = np.vstack([parents, offspring])
            combined_fit = np.concatenate([parent_fit, offspring_fit])
            best_idx = np.argsort(combined_fit)[:self.pop_size]
            return combined[best_idx], combined_fit[best_idx]
        elif self.reduction == "pure":
            return offspring[:self.pop_size], offspring_fit[:self.pop_size]
        elif self.reduction == "random":
            combined = np.vstack([parents, offspring])
            combined_fit = np.concatenate([parent_fit, offspring_fit])
            idx = self.rng.choice(len(combined), size=self.pop_size, replace=False)
            return combined[idx], combined_fit[idx]
        raise ValueError(f"Unknown reduction: {self.reduction}")

    def run(self, snapshot_every: int = 1) -> dict:
        self._init_population()

        for gen in range(self.n_generations):
            best_idx = np.argmin(self.fitness)
            self.best_history.append(float(self.fitness[best_idx]))
            self.mean_history.append(float(np.mean(self.fitness)))
            self.best_individual_history.append(self._decode(self.population[best_idx:best_idx+1])[0])

            if gen % snapshot_every == 0:
                self.population_history.append(self._decode(self.population))

            offspring_list = []
            while len(offspring_list) < self.pop_size:
                i1 = self._tournament_selection()
                i2 = self._tournament_selection()
                p1, p2 = self.population[i1].copy(), self.population[i2].copy()

                if self.rng.random() < self.crossover_prob:
                    c1, c2 = self._uniform_crossover(p1, p2)
                else:
                    c1, c2 = p1.copy(), p2.copy()

                if self.rng.random() < self.mutation_prob:
                    c1 = self._inversion_mutation(c1)
                if self.rng.random() < self.mutation_prob:
                    c2 = self._inversion_mutation(c2)

                offspring_list.extend([c1, c2])

            offspring_arr = np.array(offspring_list[:self.pop_size], dtype=np.int8)
            offspring_fit = self._evaluate(offspring_arr)

            self.population, self.fitness = self._reduce(
                self.population, self.fitness, offspring_arr, offspring_fit,
            )

        best_idx = np.argmin(self.fitness)
        self.best_history.append(float(self.fitness[best_idx]))
        self.mean_history.append(float(np.mean(self.fitness)))
        self.best_individual_history.append(self._decode(self.population[best_idx:best_idx+1])[0])
        self.population_history.append(self._decode(self.population))

        best_x = self._decode(self.population[best_idx:best_idx+1])[0]
        return {"best_x": best_x, "best_f": float(self.fitness[best_idx])}

## 4. Запуск ГА и результаты

In [ ]:
ga = BinaryGA(
    fitness_fn=schaffer_f7_vec,
    a=A, b=B,
    bits_per_var=L_BITS,
    pop_size=80,
    n_generations=200,
    crossover_prob=0.85,
    mutation_prob=0.05,
    tournament_k=3,
    reduction="elitist",
    seed=42,
)

result = ga.run(snapshot_every=1)

print(f"Лучшая найденная точка: x₁ = {result['best_x'][0]:.6f}, x₂ = {result['best_x'][1]:.6f}")
print(f"Значение функции:       f(x₁, x₂) = {result['best_f']:.6f}")
print(f"Теоретический минимум:  f(0, 0) = 0")

## 5. Визуализация сходимости

In [ ]:
fig_conv = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Лучший фитнес", "Средний фитнес"],
)

gens = list(range(len(ga.best_history)))

fig_conv.add_trace(
    go.Scatter(x=gens, y=ga.best_history, mode="lines",
               line=dict(color="royalblue", width=2), name="Лучший"),
    row=1, col=1,
)
fig_conv.add_trace(
    go.Scatter(x=gens, y=ga.mean_history, mode="lines",
               line=dict(color="darkorange", width=2), name="Средний"),
    row=1, col=2,
)

fig_conv.update_xaxes(title_text="Поколение", row=1, col=1)
fig_conv.update_xaxes(title_text="Поколение", row=1, col=2)
fig_conv.update_yaxes(title_text="f(x₁, x₂)", row=1, col=1)
fig_conv.update_yaxes(title_text="f(x₁, x₂)", row=1, col=2)
fig_conv.update_layout(
    title="Сходимость генетического алгоритма (бинарное кодирование)",
    width=1000, height=450, template="plotly_white",
)
fig_conv.write_html(OUTPUTS / "02_convergence.html")
fig_conv.show()

## 6. Анимация популяции на контурном графике

In [ ]:
snapshots = ga.population_history
step = max(1, len(snapshots) // 40)
selected = list(range(0, len(snapshots), step))
if (len(snapshots) - 1) not in selected:
    selected.append(len(snapshots) - 1)

frames = []
for idx in selected:
    pop = snapshots[idx]
    frames.append(go.Frame(
        data=[go.Scatter(
            x=pop[:, 0], y=pop[:, 1],
            mode="markers",
            marker=dict(size=5, color="red", opacity=0.7),
        )],
        name=str(idx),
    ))

first_pop = snapshots[selected[0]]

fig_anim = go.Figure(
    data=[
        go.Contour(
            x=x_grid, y=x_grid, z=Z,
            contours=dict(start=0, end=20, size=1),
            colorscale="Viridis",
            showscale=False,
        ),
        go.Scatter(
            x=first_pop[:, 0], y=first_pop[:, 1],
            mode="markers",
            marker=dict(size=5, color="red", opacity=0.7),
            name="Популяция",
        ),
    ],
    frames=frames,
)

fig_anim.update_layout(
    title="Динамика популяции по поколениям",
    xaxis_title="x₁", yaxis_title="x₂",
    width=750, height=700,
    template="plotly_white",
    updatemenus=[dict(
        type="buttons", showactive=False,
        x=0.05, y=1.12,
        buttons=[
            dict(label="▶ Пуск", method="animate",
                 args=[None, {"frame": {"duration": 200, "redraw": True},
                              "fromcurrent": True}]),
            dict(label="⏸ Пауза", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False},
                                "mode": "immediate"}]),
        ],
    )],
    sliders=[dict(
        active=0,
        steps=[dict(args=[[str(idx)], {"frame": {"duration": 200, "redraw": True},
                                       "mode": "immediate"}],
                    label=str(idx), method="animate")
               for idx in selected],
        x=0.05, len=0.9,
        currentvalue=dict(prefix="Поколение: "),
    )],
)
fig_anim.write_html(OUTPUTS / "03_population_animation.html")
fig_anim.show()

## 7. Сравнение стратегий редукции

Исследуем три стратегии формирования нового поколения:

| Стратегия | Описание |
|-----------|----------|
| **Элитарная** | Из объединения родителей и потомков отбираются лучшие |
| **Полная замена** | Потомки полностью заменяют родителей |
| **Случайная замена** | Из объединения случайно выбираются $N$ особей |

In [ ]:
reduction_results = {}

for strategy in ["elitist", "pure", "random"]:
    ga_r = BinaryGA(
        fitness_fn=schaffer_f7_vec,
        a=A, b=B,
        bits_per_var=L_BITS,
        pop_size=80,
        n_generations=200,
        crossover_prob=0.85,
        mutation_prob=0.05,
        tournament_k=3,
        reduction=strategy,
        seed=42,
    )
    res = ga_r.run()
    reduction_results[strategy] = {
        "result": res,
        "best_history": ga_r.best_history,
        "mean_history": ga_r.mean_history,
    }
    label = {"elitist": "Элитарная", "pure": "Полная замена", "random": "Случайная"}[strategy]
    print(f"{label:20s}: f = {res['best_f']:.6f}  x = ({res['best_x'][0]:+.4f}, {res['best_x'][1]:+.4f})")

In [ ]:
colors = {"elitist": "royalblue", "pure": "crimson", "random": "forestgreen"}
labels = {"elitist": "Элитарная", "pure": "Полная замена", "random": "Случайная"}

fig_red = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Лучший фитнес", "Средний фитнес"],
)

for strategy, data in reduction_results.items():
    gens_r = list(range(len(data["best_history"])))
    fig_red.add_trace(
        go.Scatter(x=gens_r, y=data["best_history"], mode="lines",
                   line=dict(color=colors[strategy], width=2),
                   name=labels[strategy], legendgroup=strategy),
        row=1, col=1,
    )
    fig_red.add_trace(
        go.Scatter(x=gens_r, y=data["mean_history"], mode="lines",
                   line=dict(color=colors[strategy], width=2, dash="dash"),
                   name=labels[strategy] + " (сред.)", legendgroup=strategy,
                   showlegend=False),
        row=1, col=2,
    )

fig_red.update_xaxes(title_text="Поколение", row=1, col=1)
fig_red.update_xaxes(title_text="Поколение", row=1, col=2)
fig_red.update_yaxes(title_text="f(x₁, x₂)", row=1, col=1)
fig_red.update_yaxes(title_text="f(x₁, x₂)", row=1, col=2)
fig_red.update_layout(
    title="Сравнение стратегий редукции",
    width=1000, height=450, template="plotly_white",
)
fig_red.write_html(OUTPUTS / "04_reduction_comparison.html")
fig_red.show()

## 8. Анализ влияния гиперпараметров

Исследуем влияние следующих параметров:
- **Размер популяции:** 20, 40, 80, 160
- **Вероятность кроссовера:** 0.5, 0.7, 0.85, 0.95
- **Вероятность мутации:** 0.01, 0.05, 0.1, 0.2
- **Размер турнира:** 2, 3, 5, 7

In [ ]:
def run_experiment(**kwargs) -> float:
    defaults = dict(
        fitness_fn=schaffer_f7_vec, a=A, b=B, bits_per_var=L_BITS,
        pop_size=80, n_generations=200, crossover_prob=0.85,
        mutation_prob=0.05, tournament_k=3, reduction="elitist", seed=42,
    )
    defaults.update(kwargs)
    ga_e = BinaryGA(**defaults)
    res = ga_e.run()
    return res["best_f"]


param_configs = {
    "Размер популяции": ("pop_size", [20, 40, 80, 160]),
    "P кроссовера": ("crossover_prob", [0.5, 0.7, 0.85, 0.95]),
    "P мутации": ("mutation_prob", [0.01, 0.05, 0.1, 0.2]),
    "Размер турнира": ("tournament_k", [2, 3, 5, 7]),
}

param_results = {}
for title, (param_name, values) in param_configs.items():
    fits = []
    for v in values:
        f = run_experiment(**{param_name: v})
        fits.append(f)
    param_results[title] = (values, fits)
    print(f"{title}:")
    for v, f in zip(values, fits):
        print(f"  {param_name}={v}  →  f = {f:.6f}")

In [ ]:
fig_params = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(param_results.keys()),
)

positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
bar_colors = ["royalblue", "darkorange", "forestgreen", "crimson"]

for (title, (vals, fits)), (r, c), color in zip(param_results.items(), positions, bar_colors):
    fig_params.add_trace(
        go.Bar(
            x=[str(v) for v in vals], y=fits,
            marker_color=color, name=title, showlegend=False,
        ),
        row=r, col=c,
    )
    fig_params.update_yaxes(title_text="f(x₁, x₂)", row=r, col=c)

fig_params.update_layout(
    title="Влияние гиперпараметров на качество оптимизации",
    width=1000, height=700, template="plotly_white",
)
fig_params.write_html(OUTPUTS / "05_param_analysis.html")
fig_params.show()

## 9. Итоговая таблица результатов

In [ ]:
table_rows = []

for strategy in ["elitist", "pure", "random"]:
    r = reduction_results[strategy]["result"]
    table_rows.append([
        labels[strategy],
        f"{r['best_x'][0]:+.6f}",
        f"{r['best_x'][1]:+.6f}",
        f"{r['best_f']:.6f}",
    ])

for title, (vals, fits) in param_results.items():
    best_idx = int(np.argmin(fits))
    table_rows.append([
        f"Лучш. {title} = {vals[best_idx]}",
        "—", "—",
        f"{fits[best_idx]:.6f}",
    ])

fig_table = go.Figure(data=go.Table(
    header=dict(
        values=["Конфигурация", "x₁", "x₂", "f(x₁, x₂)"],
        fill_color="royalblue",
        font=dict(color="white", size=13),
        align="center",
    ),
    cells=dict(
        values=list(zip(*table_rows)),
        fill_color="lavender",
        align="center",
        font=dict(size=12),
        height=30,
    ),
))
fig_table.update_layout(
    title="Сводная таблица результатов",
    width=900, height=450, template="plotly_white",
)
fig_table.write_html(OUTPUTS / "06_results_table.html")
fig_table.show()

## Выводы

1. **Бинарное кодирование.** Реализовано бинарное представление двух вещественных переменных $x_1, x_2 \in [-100, 100]$ с точностью дискретизации $\approx 1.9 \times 10^{-4}$ при длине хромосомы 20 бит на переменную.

2. **Операторы ГА.** Турнирная селекция ($k=3$), равномерный кроссовер и инверсионная мутация обеспечили эффективный поиск глобального минимума функции Шаффера F7.

3. **Стратегии редукции.** Элитарная стратегия показала наилучшую сходимость: она сохраняет лучшие решения и монотонно улучшает значение фитнеса. Полная замена теряет лучшие решения, а случайная — сходится медленнее.

4. **Гиперпараметры.** Увеличение размера популяции и размера турнира повышает качество решения. Оптимальные вероятности кроссовера и мутации находятся в диапазонах $P_c \approx 0.85$ и $P_m \approx 0.05$.

5. **Результат.** ГА успешно находит область глобального минимума $f(0, 0) = 0$ функции Шаффера F7, несмотря на большое количество локальных минимумов.